#  Project Part I — Language and Grammar Engineering

## 1. Desenho da linguagem

### 1.1 Contexto

Na unidade curricular de Visualização em Tempo Real (VTR) os trabalhos práticos são feitos no **Nau**, um motor gráfico desenvolvido na Universidade do Minho. Um motor gráfico é o programa que pega numa descrição de uma cena 3D (que objetos existem, onde estão, de onde se olha para eles, de onde vem a luz) e a desenha no ecrã, usando a placa gráfica.

No Nau não se programa o motor. Cada trabalho é um conjunto de ficheiros de texto que o motor lê:

| Ficheiro | O que descreve |
|---|---|
| projeto (`.xml`) | as cenas (os objetos 3D), a câmara, as luzes e a ordem pela qual se desenha |
| biblioteca de materiais (`.mlib`) | o aspeto de cada objeto (cor, brilho) e o programa que o desenha |
| shaders (`.vert`, `.frag`) | pequenos programas, escritos em GLSL (uma linguagem parecida com C), que correm na placa gráfica e calculam a cor de cada ponto |

Para perceber o resto do relatório chega saber o seguinte. Os **shaders** precisam de receber valores vindos de fora, como a cor do objeto, a direção da luz ou a posição da câmara. Essas "entradas" chamam-se *uniforms* e cada uma tem um nome, por exemplo `l_dir` para a direção da luz. É o ficheiro `.mlib` que diz ao Nau que valor deve pôr em cada uniform, referindo-o **pelo nome**.

### 1.2 O problema

O projeto mais simples da aula 2 de VTR desenha uma chaleira, vista por uma câmara e iluminada por uma luz. Para isso são precisas **175 linhas em quatro ficheiros**: 66 de XML no projeto, 72 no `.mlib` e 33 nos dois shaders. Um excerto mostra o tipo de texto envolvido:

```xml
<camera name="MainCamera">
    <viewport name="MainViewport" />
    <POSITION x="1.46" y="1.05" z="1.75" />
    <LOOK_AT_POINT x="0" y="0" z="0" />
</camera>
...
<valueof uniform="l_dir" type="LIGHT" component="DIRECTION" id="0" />
```

Escrever estes ficheiros à mão tem quatro problemas:

1. **É repetitivo.** Linhas como a do `valueof` acima repetem-se, quase iguais, em todos os materiais.
2. **Tudo se liga por nomes, em ficheiros diferentes.** A câmara refere o viewport pelo nome, o passe de desenho refere a câmara, as cenas e as luzes pelo nome, e o `.mlib` refere os uniforms do shader pelo nome. Nada garante que esses nomes coincidem.
3. **Os erros são silenciosos.** Se um nome estiver mal escrito, o Nau não mostra nenhuma mensagem: o valor fica simplesmente a zero e o objeto aparece preto, ou nem aparece. Encontrar a causa obriga a comparar os três ficheiros linha a linha.
4. **Os shaders são quase sempre os mesmos.** Nas aulas 2 e 3 usam-se poucos efeitos (iluminação calculada nos vértices, iluminação calculada em cada píxel com brilho, iluminação por patamares, texturas de dia/noite/brilho/nuvens). Cada projeto copia os mesmos `.vert` e `.frag` de outro e muda os nomes dos uniforms, o que volta a criar os problemas 2 e 3.

Do ponto de vista de Engenharia Gramatical, o problema é que o XML só verifica a **forma** do texto (as etiquetas estão bem abertas e fechadas), mas não o **significado** (se aquilo a que um nome se refere existe e é do tipo certo).

### 1.3 Objetivo da linguagem

A **NSL (Nau Scene Language)** é uma linguagem de domínio específico para descrever cenas 3D do Nau de forma curta e legível. Um programa NSL é traduzido por um compilador para todos os ficheiros de texto que o Nau lê: o projeto (`.xml`), a biblioteca de materiais (`.mlib`) e os shaders (`.vert`, `.frag`). Os modelos 3D (`.obj`) e as imagens das texturas continuam a ser ficheiros externos, referidos pelo caminho. O motor não é alterado.

Na NSL não se escrevem shaders. Cada material diz **que efeito quer** (por exemplo, `lighting phong` ou uma camada de textura para a noite) e o compilador gera o GLSL correspondente a partir de modelos pré-definidos, juntamente com as ligações entre o shader e o motor. Como é o compilador que escreve os dois lados dessas ligações, os nomes dos uniforms nunca chegam ao utilizador.

A linguagem pretende:

- **reduzir a escrita:** cada coisa (textura, material, câmara, luz, cena) declara-se uma vez, num bloco com um nome, e tudo o que é repetitivo, incluindo os shaders, é gerado automaticamente;
- **apanhar os erros antes de chegar ao Nau:** como todos os nomes ficam declarados no mesmo programa, é possível verificar que cada referência existe e é do tipo certo (por exemplo, um passe só pode desenhar cenas, não câmaras) e que as opções de cada material são compatíveis entre si (por exemplo, o brilho só existe na iluminação `phong`);
- **permitir cenas parametrizáveis:** variáveis, ciclos, condições e *templates* permitem gerar muitos objetos ou várias versões de uma cena a partir de poucas linhas.

**Âmbito.** A NSL cobre o que é pedido nas aulas 2 (iluminação) e 3 (texturas) de VTR: os efeitos disponíveis são exatamente os dessas aulas. Funcionalidades mais avançadas do Nau, como sombras, desenho em vários passes encadeados ou shaders escritos pelo utilizador, ficam fora desta versão e são referidas como trabalho futuro.

**Utilizadores.** Os alunos de VTR e, em geral, quem quer montar no Nau as cenas das aulas, ou variações delas, sem escrever XML nem GLSL: pessoas que sabem o que querem ver no ecrã (que objetos, com que cores, texturas e iluminação), mas não querem perder tempo com a configuração.

### 1.4 Conceitos principais

| Conceito | Em palavras simples | Em NSL |
|---|---|---|
| Textura | uma imagem "colada" à superfície de um objeto (por exemplo, o mapa da Terra numa esfera) | `texture earth = "earthDay.jpg" mipmap;` |
| Material | o aspeto de um objeto: cor, iluminação e camadas de textura | `material amarelo { ... }` |
| Iluminação | a forma como a luz é calculada no objeto; o compilador gera o shader correspondente | `lighting per_vertex;`, `lighting phong { shininess 100; }`, `lighting toon { levels 4; }`, `lighting none;` |
| Camada | uma textura com um papel definido no material: cor de dia, zonas que brilham, luzes de noite, nuvens | `layer day earth;`, `layer specular spec;`, `layer night eNight;`, `layer clouds noise speed 0.0001;` |
| Cena | um grupo de objetos 3D, carregados de um ficheiro ou gerados (esfera, caixa, grelha) | `scene mundo { ... }` |
| Objeto e transformações | uma forma, com a sua posição, tamanho e rotação | `box caixa : azul { translate (2, 0, 0); }` |
| Câmara | o ponto de onde se olha e para onde se olha | `camera cam { ... }` |
| Luz | a direção de onde vem a iluminação | `light sol directional { ... }` |
| Passe | uma ordem de desenho: estas cenas, com esta câmara e estas luzes | `pass render { ... }` |
| Variável | um valor com nome, usado em expressões | `let raio = 4;` |
| Ciclo e condição | gerar objetos repetidos ou escolher entre alternativas | `for i in 0..n { ... }`, `if ... { } else { }` |
| Template | um grupo de objetos com parâmetros, reutilizável | `template pilar(h: num, mat: material) { ... }` |

Os quatro modos de iluminação correspondem aos shaders das aulas: `per_vertex` calcula a luz só nos vértices dos triângulos (o `simple` da aula 2), `phong` calcula-a em cada píxel e acrescenta o brilho especular (o `v4`), `toon` arredonda a intensidade a um número fixo de patamares, dando um aspeto de desenho animado, e `none` mostra a cor sem luz.

### 1.5 Como as construções servem o objetivo

**Declarações com nome.** Texturas, materiais, cenas, câmaras e luzes são declarados com um nome e usados depois através dele. Isto continua a ser uma ligação por nomes, tal como no XML, mas agora todas as declarações estão no mesmo programa. O compilador pode por isso construir uma tabela de símbolos e verificar cada referência.

**Efeitos em vez de shaders.** O material descreve o resultado pretendido (`lighting`, `layer`) e não a forma de o calcular. A partir dessa descrição o compilador escolhe o modelo de shader, gera os ficheiros `.vert` e `.frag`, e escreve no `.mlib` o valor de cada uniform e o número de cada unidade de textura. Assim desaparecem duas fontes de erros silenciosos: nomes de uniforms que não coincidem e números de unidades de textura trocados. Em troca, as combinações de opções passam a ter regras (por exemplo, uma camada `specular` só faz sentido com `lighting phong`), que o compilador verifica.

**Blocos aninhados.** Um material, uma cena ou uma câmara é um bloco `{ ... }` com propriedades lá dentro. Uma cena contém objetos e cada objeto contém as suas transformações. Esta estrutura em árvore corresponde à estrutura da própria cena.

**Variáveis, expressões, ciclos, condições e templates.** O Nau não tem nada disto: o XML só aceita valores fixos. Na NSL, estas construções são **avaliadas pelo compilador**, que as resolve antes de gerar os ficheiros. Um ciclo que cria oito pilares produz oito elementos `<geometry>` no XML, cada um com a sua posição já calculada. Uma condição escolhe que objetos ou que propriedades entram no resultado. O ficheiro gerado é sempre XML normal do Nau.

O `if` tem usos concretos no domínio, por exemplo:

- alternar entre modos de iluminação (na aula 2 comparam-se `per_vertex`, `phong` e `toon`) mudando uma única variável;
- mostrar elementos de apoio, como os eixos, só enquanto se testa a cena;
- dentro de um ciclo, alternar materiais ou saltar posições.

**Âmbitos.** As declarações de topo são globais. Um ciclo `for` abre um âmbito novo: a variável do ciclo e os `let` declarados dentro do bloco só existem nesse bloco. O mesmo acontece com os parâmetros de um `template`, que têm um tipo declarado (`num`, `material`, `color`). Os blocos `if`/`else` também abrem um âmbito próprio.

**Relação com os requisitos mínimos do enunciado:**

| Requisito | Onde aparece na NSL |
|---|---|
| Variáveis e declarações | `let`, e as declarações de `texture`, `material`, `scene`, `camera`, `light`, `pass`, `template` |
| Âmbitos | global, corpo do `for`, ramos do `if`, corpo do `template` |
| Expressões e valores | expressões aritméticas e de comparação em posições, ângulos, cores, parâmetros de iluminação e condições |
| Construções compostas e aninhadas | cena → objetos → transformações; material → iluminação e camadas; `for` e `if` dentro de cenas e materiais, e uns dentro dos outros |
| Ramificação e repetição | `if`/`else` e `for` |
| Propriedades que a sintaxe não determina | ver lista abaixo |

**Propriedades semânticas** (não podem ser verificadas pela gramática independente de contexto):

- **Nomes:** todo o nome usado está declarado e visível no âmbito onde é usado, e não há duas declarações com o mesmo nome no mesmo âmbito.
- **Tipos das referências:** `draw` só aceita cenas, `camera` só aceita câmaras, `layer` só aceita texturas, e os argumentos de um `template` têm o tipo declarado nos parâmetros (`draw cam` é um erro, porque `cam` é uma câmara).
- **Valores válidos:** `near > 0`, `far > near`, `shininess > 0`, `levels >= 2`, componentes de cor entre 0 e 1.
- **Coerência do material:**
  - `shininess` só pode aparecer em `lighting phong` e `levels` só em `lighting toon`;
  - as camadas `specular` exigem `lighting phong`, e as camadas `night` exigem um modo com luz (não `none`);
  - cada tipo de camada aparece no máximo uma vez por material, e `specular`, `night` e `clouds` exigem uma camada `day`;
  - um material tem no máximo um `lighting` (por omissão, `per_vertex`).
- **Templates:** um template não pode chamar-se a si próprio, direta ou indiretamente.
- **Atribuição de unidades de textura:** cada camada de um material recebe um número de unidade diferente (0, 1, 2, ...), pela ordem em que aparece. Este valor não é verificado, mas **calculado** pelo compilador.

### 1.6 Exemplos de programas

Os exemplos 1, 2 e 4 são os projetos das aulas de VTR reescritos em NSL, **completos**: o programa NSL produz o mesmo resultado que todos os ficheiros originais juntos. Por baixo de cada exemplo estão os ficheiros originais, que se podem expandir para comparar. O exemplo 3 não vem das aulas e mostra o que a NSL acrescenta.

| Exemplo | Original (aulas de VTR) | NSL |
|---|---|---|
| 1. Iluminação simples (aula 2) | 175 linhas em 4 ficheiros (`.xml`, `.mlib`, `.vert`, `.frag`) | 28 linhas, 1 ficheiro |
| 2. Iluminação por píxel (aula 2) | 191 linhas em 4 ficheiros (o `.mlib` é partilhado com o exemplo 1) | 34 linhas, 1 ficheiro, com os três modos |
| 4. Texturas da Terra (aula 3) | 232 linhas em 4 ficheiros, mais a biblioteca de ruído `noise.frag` (132 linhas) | 33 linhas, 1 ficheiro |

As linhas em branco e os comentários estão incluídos nas duas colunas.

**Exemplo 1: o projeto da aula 2 (`simple.xml`).** Uma chaleira, vista pela câmara e iluminada pelo sol, com a luz calculada nos vértices.

```
project "Simple";

scene teapot_l = model "models/teapot.3ds" unitize swap_yz;
scene teapot_s = model "models/teapot.obj" unitize;
scene al       = model "models/al.obj"     unitize;
scene plane    = grid length 20 divisions 2;

material Yellow { diffuse rgb(1.0, 0.87, 0.0); }
material Blue   { diffuse rgb(0.0, 0.34, 0.72); }

camera MainCamera {
    position (1.46, 1.05, 1.75);
    look_at  (0, 0, 0);
    near     0.01;
}

light Sun directional {
    direction (2, 1, 2);
}

background white;

pass pass1 {
    draw   teapot_s;
    camera MainCamera;
    lights Sun;
    lighting per_vertex for all;   // ilumina tudo desta forma, mantendo as cores do modelo
}
```

Algumas observações sobre o original, que a tradução torna visíveis:

- O projeto declara quatro cenas (`teapot_l`, `teapot_s`, `al`, `plane`) mas só desenha `teapot_s`. A biblioteca declara os materiais `Yellow` e `Blue`, que nenhum objeto usa. O programa NSL mantém estas declarações para ser equivalente, e o compilador pode avisar que não são usadas.
- No original, o `.xml` escolhe o shader escrevendo `fromLibrary="simple" fromMaterial="simple"`, dois nomes que têm de coincidir com o `.mlib`. O `.mlib`, por sua vez, refere os uniforms `m_pvm`, `m_view`, `m_normal`, `l_dir` e `diffuse`, que têm de coincidir com o `.vert`. Na NSL estas ligações desaparecem: `lighting per_vertex` gera os shaders e as ligações juntos.

<details>
<summary><b>Original:</b> <code>simple.xml</code> (67 linhas)</summary>

```xml
<project name="Simple" >
    <assets>
        <scenes>
            <scene name="teapot_l" param="SWAP_YZ UNITIZE">
                <file name="models/teapot.3ds"/>
            </scene>

            <scene name="teapot_s" param="UNITIZE">
                <file name="models/teapot.obj"/>
            </scene>

            <scene name="al" param="UNITIZE">
                <file name="models/al.obj"/>
            </scene>
            
            <scene name="plane" >
                <geometry name = "Grid" type = "GRID" LENGTH="20"  DIVISIONS="2"></geometry>
            </scene>
        </scenes>
        
        <viewports>
            <viewport name="MainViewport">
                <CLEAR_COLOR r="1.0" g="1.0" b="1.0" />
            </viewport>
        </viewports>        

        <cameras>
            <camera name="MainCamera" >
                <viewport name="MainViewport" />
                <POSITION x="1.46" y="1.05" z="1.75"  />
                <LOOK_AT_POINT x = "0" y="0" z="0" />
                <NEAR value = "0.01" />
            </camera>
        </cameras>
        
        <lights>
            <light name="Sun">
                <DIRECTION x="2" y="1" z="2" />
            </light>
        </lights>

        <materialLibs>
            <materialLib filename="simple.mlib"/>
        </materialLibs>

    </assets>
    
    <pipelines>
        <pipeline name="simple" default="true">
            <pass class="default" name="pass1">
                <scenes>
                    <scene name="teapot_s" />
                </scenes>
                <camera name="MainCamera" />
                <lights>
                    <light name="Sun" />
                </lights>
                <injectionMaps>
                    <map toMaterial="*">
                        <shader fromLibrary="simple" fromMaterial="simple" />
                    </map>
                </injectionMaps>
            </pass>
        </pipeline>
    </pipelines>

</project>
```

</details>

<details>
<summary><b>Original:</b> <code>simple.mlib</code> (73 linhas)</summary>

```xml
<?xml version="1.0" ?>
<materialLib name="simple">

    <shaders>
        <shader name="simple" vs="shaders/simple.vert" ps="shaders/simple.frag" />
        <shader name="v4" vs="shaders/v4.vert" ps="shaders/v4.frag" />

    </shaders>

    <materials>

        <material name="Yellow">
            <color>
                <DIFFUSE r="1.0" g="0.87" b="0.0" />
            </color>
        </material>

        <material name="Blue">
            <color>
                <DIFFUSE r="0.0" g="0.34" b="0.72" />
            </color>
        </material>


        <material name="simple">
            <shader name="simple" >
                <values>
                    <valueof uniform="m_pvm" type="RENDERER" component="PROJECTION_VIEW_MODEL" />
                    <valueof uniform="m_view" type="RENDERER" component="VIEW" />
                    <valueof uniform="m_normal" type="RENDERER" component="NORMAL" />
                    <valueof uniform="l_dir" type="LIGHT" component="DIRECTION" id="0" />
                    <valueof uniform="diffuse" type="COLOR" component="DIFFUSE" />
                </values>
            </shader>
        </material>

        <material name="v4">
            <color>
                <DIFFUSE r="1.0" g="0.87" b="0.0" a="1.0" />
                <SHININESS value="100" />
            </color>
            <shader name="v4" >
                <values>
                    <valueof uniform="m_pvm"
                            type="RENDERER"
                            component="PROJECTION_VIEW_MODEL" />
                    <valueof uniform="m_vm"
                            type="RENDERER"
                            component="VIEW_MODEL" />
                    <valueof uniform="m_v"
                            type="RENDERER"
                            component="VIEW" />
                    <valueof uniform="m_n"
                            type="RENDERER"
                            component="NORMAL" />
                    <valueof uniform="ld"
                            type="LIGHT"
                            component="DIRECTION" id="0" />
                    
                    <valueof uniform="diffuse"
                            type="COLOR"
                            component="DIFFUSE" />
                    <valueof uniform="shininess"
                            type="COLOR"
                            component="SHININESS" />
                </values>
            </shader>       
        </material>


    </materials>

</materialLib>
```

</details>

<details>
<summary><b>Original:</b> <code>shaders/simple.vert</code> (25 linhas)</summary>

```glsl
#version 460

uniform mat4 m_pvm, m_view;
uniform mat3 m_normal;
uniform vec4 diffuse;

uniform vec4 l_dir;

in vec4 position;
in vec3 normal;

out vec4 color_V;

void main() {

    vec3 n = normalize(m_normal * normal);
    vec4 l4 = m_view * l_dir;
    vec3 l = normalize(vec3(l4));

    float intensity = max(0.0, dot(l,n));

    color_V = intensity * diffuse; 

    gl_Position = m_pvm * position;
}
```

</details>

<details>
<summary><b>Original:</b> <code>shaders/simple.frag</code> (10 linhas)</summary>

```glsl
#version 460

in vec4 color_V;

out vec4 color;

void main() {

    color = color_V;
}
```

</details>

**Exemplo 2: o projeto da aula 2 com iluminação por píxel (`simple4.xml`).** A mesma aula compara formas diferentes de iluminar o objeto. No original, mudar de modo obriga a editar `fromMaterial="v4"` no `.xml`, e cada modo tem o seu par de shaders com nomes de uniforms diferentes (`m_v`, `m_n` e `ld` no `v4`, mas `m_view`, `m_normal` e `l_dir` no `simple`). Na NSL basta mudar o valor de `modo`, e o compilador gera apenas o shader do modo escolhido.

```
project "Simple";

let modo = 2;          // 1 = por vértice (simple), 2 = phong (v4), 3 = toon

scene rb         = model "models/rainbowBench.obj" unitize;
scene al         = model "models/al.obj"           unitize;
scene teapot_obj = model "models/teapot.obj"       unitize;
scene teapot_3ds = model "models/teapot.3ds"       unitize swap_yz;
scene plane      = grid length 20 divisions 2;

camera MainCamera {
    position (1.46, 1.05, 1.75);
    look_at  (0, 0, 0);
    near     0.1;
}

light Sun directional {
    direction (-2, -1, -2);
}

background rgb(0.15, 0.25, 0.45);

pass pass1 {
    draw   teapot_3ds;
    camera MainCamera;
    lights Sun;
    if modo == 2 {
        lighting phong { shininess 100; } for all;
    } else if modo == 3 {
        lighting toon { levels 4; } for all;
    } else {
        lighting per_vertex for all;
    }
}
```

<details>
<summary><b>Original:</b> <code>simple4.xml</code> (72 linhas)</summary>

```xml

<project name="Simple" >
    <assets>
        <scenes>
            <scene name="rb" param="UNITIZE">
                <file name="models/rainbowBench.obj"/>
            </scene>
            <scene name="al" param="UNITIZE">
                <file name="models/al.obj"/>
            </scene>
            <scene name="teapot_obj" param="UNITIZE">
                <file name="models/teapot.obj"/>
            </scene>
            <scene name="teapot_3ds" param="UNITIZE SWAP_YZ">
                <file name="models/teapot.3ds"/>
            </scene>
            <scene name="plane" >
                <geometry name = "Grid" type = "GRID" LENGTH="20"  DIVISIONS="2"></geometry>
            </scene>

        </scenes>
        
        <viewports>
            <viewport name="MainViewport">
                <CLEAR_COLOR r="0.15" g="0.25" b="0.45" />
            </viewport>
        </viewports>        

        <cameras>
            <camera name="MainCamera" >
                <viewport name="MainViewport" />
                <POSITION x="1.46" y="1.05" z="1.75"  />
                <LOOK_AT_POINT x = "0" y="0" z="0" />
                <NEAR value="0.1" />
            </camera>
        </cameras>
        
        <lights>
            <light name="Sun">
                <DIRECTION x="-2" y="-1" z="-2" />
            </light>
        </lights>

        <!-- adding a material lib -->
        <materialLibs>
            <materialLib filename="simple.mlib"/>
        </materialLibs>


    </assets>
    
    <pipelines>
        <pipeline name="simple" default="true">
            <pass class="default" name="pass1">
                <scenes>
                    <scene name="teapot_3ds" />
                </scenes>
                <camera name="MainCamera" />
                <lights>
                    <light name="Sun" />
                </lights>
                <injectionMaps>
                    <map toMaterial="*">
                        <shader fromLibrary="simple" fromMaterial="v4" />
                    </map>   
                </injectionMaps>
            </pass>
        </pipeline>
    </pipelines>


</project>
```

</details>

<details>
<summary><b>Original:</b> <code>shaders/v4.vert</code> (21 linhas)</summary>

```glsl
#version 460

in vec4 position; // local
in vec3 normal;   // local

uniform mat4 m_pvm, m_vm, m_v;
uniform mat3 m_n;
uniform vec4 ld;  // direcao da luz (espaco global)

out vec3 nn;      // normal (espaco camara)
out vec3 ldn;     // direcao para a luz (espaco camara)
out vec3 e;       // vetor para o olho (espaco camara)

void main() {

    nn  = normalize(m_n * normal);
    ldn = normalize(vec3(m_v * -ld));
    e   = -vec3(m_vm * position);

    gl_Position = m_pvm * position;
}
```

</details>

<details>
<summary><b>Original:</b> <code>shaders/v4.frag</code> (25 linhas)</summary>

```glsl
#version 460

in vec3 nn, ldn, e;

uniform vec4 diffuse;
uniform float shininess;

out vec4 color;

void main() {

    vec3 n = normalize(nn);
    vec3 l = normalize(ldn);

    float intensity = max(0.0, dot(n, l));
    vec4 specular = vec4(0.0);

    if (intensity > 0.0) {
        vec3 h = normalize(normalize(e) + l);
        float spec = max(0.0, dot(h, n));
        specular = vec4(1.0) * pow(spec, shininess);
    }

    color = max(0.25, intensity) * diffuse + specular;
}
```

</details>

O `.mlib` usado é o mesmo do exemplo 1 (`simple.mlib`), onde está o material `v4` com `shininess` 100.

**Exemplo 3: uma cena gerada com template, ciclo e condição.** Este exemplo não vem das aulas: mostra o que a NSL acrescenta ao Nau. Oito pilares dispostos em círculo, alternando a altura e o material. O compilador calcula o ângulo e a posição de cada um e gera oito objetos no XML, com nomes únicos. Escrito à mão, o mesmo resultado exigiria oito elementos `<geometry>` com posições e rotações calculadas à parte.

```
let n    = 8;
let raio = 4;

material azul   { diffuse rgb(0.0, 0.34, 0.72); lighting phong { shininess 60; } }
material branco { diffuse white;                lighting per_vertex; }

template pilar(h: num, mat: material) {
    box p : mat {
        scale     (0.5, h, 0.5);
        translate (0, h / 2, 0);
    }
}

scene templo {
    for i in 0..n {                 // i = 0, 1, ..., n-1
        let ang = i * 360 / n;
        if i % 2 == 0 {
            pilar(3, azul)   { rotate ang around (0, 1, 0); translate (raio, 0, 0); }
        } else {
            pilar(2, branco) { rotate ang around (0, 1, 0); translate (raio, 0, 0); }
        }
    }
}

scene chao = grid length 20 divisions 2;

camera cam {
    position (8, 6, 8);
    look_at  (0, 1, 0);
}

light sol directional { direction (2, 1, 2); }

pass render {
    draw   templo, chao;
    camera cam;
    lights sol;
}
```

Neste exemplo há vários erros possíveis que o XML do Nau nunca detetaria e que a NSL deteta antes de gerar os ficheiros. Por exemplo: escrever `pilar(3, azl)` (material inexistente), `pilar(azul, 3)` (argumentos com o tipo errado), usar `ang` fora do ciclo (fora do âmbito), escrever `draw templo, cam` (`cam` não é uma cena) ou pôr `shininess` no material `branco` (só existe em `phong`). Estas verificações são tratadas na secção de semântica.

**Exemplo 4: o projeto de texturas da aula 3 (`textureModel.xml`).** Uma esfera com o aspeto da Terra: a superfície de dia, as zonas que refletem a luz (o mar), as luzes das cidades no lado de noite e nuvens geradas por ruído que se movem com o tempo.

No Nau, cada imagem é ligada ao shader através de um número de "unidade de textura", que tem de ser escrito em dois sítios diferentes do `.mlib`. Basta trocar um número para a esfera aparecer com a imagem errada, sem qualquer aviso. Em NSL, o material diz apenas que papel tem cada imagem (`layer night eNight;`), e é o compilador que atribui os números e gera o shader que as combina.

```
project "Texture Model";

texture earth   = "otherTextures/earthDay.jpg"    mipmap;
texture spec    = "otherTextures/earthSpec.jpg"   mipmap;
texture eNight  = "otherTextures/earthLights.jpg" mipmap;
texture eClouds = "otherTextures/earthClouds.jpg" mipmap;   // declarada no original, mas nunca usada

material textured {
    lighting phong { shininess 128; }
    layer day      earth;
    layer specular spec;                  // só brilha onde a imagem é clara (o mar)
    layer night    eNight;                // aparece onde não chega luz
    layer clouds   noise speed 0.0001;    // nuvens procedimentais, animadas com o tempo
}

scene Sphere = model "models/sphere.obj" unitize;

camera MainCamera {
    position  (0, 0, 3);
    direction (0, 0, -1);                 // direção para onde olha, em vez de um ponto
    near      0.1;
}

light Sun directional { direction (2, -1, -2); }

background black;

pass texture {
    draw   Sphere;
    camera MainCamera;
    lights Sun;
    material textured for all;            // substitui os materiais do modelo por este
}
```

Observações sobre o original:

- A textura `eClouds` é declarada e ligada à unidade 3, mas o shader não a usa: a linha que a lê está comentada e as nuvens são geradas por ruído. Em NSL, a textura fica declarada mas nenhuma camada a refere, e o compilador pode avisar disso.
- O `.xml` refere `models/Sphere.obj`, mas o ficheiro chama-se `sphere.obj`. Em Windows funciona, mas em Linux o Nau não encontra o modelo. Um compilador NSL pode verificar se os ficheiros referidos existem.
- O `shininess` não tem `<valueof>` no `.mlib`: o valor 128 vem de um valor por omissão escrito dentro do `texture.frag`. Na NSL o valor fica à vista no material.

<details>
<summary><b>Original:</b> <code>textureModel.xml</code> (58 linhas)</summary>

```xml
<?xml version="1.0" ?>
<project name="Texture Model" >
    <assets>
    
        <scenes>
            <scene name="Sphere" param="UNITIZE">
                <file name="models/Sphere.obj"/>
            </scene>
        </scenes>
        
        <viewports>
            <viewport name="MainViewport">
                <CLEAR_COLOR r="0.0" g="0.0" b="0.0" />
            </viewport>
        </viewports>
        
        <cameras>
            <camera name="MainCamera" >
                <viewport name="MainViewport" />
                <POSITION x="0" y="0" z="3" />
                <VIEW x="0" y="0" z="-1.0" />
                <NEAR value="0.1" />
            </camera>
        </cameras>
        
        <lights>
            <light name="Sun">
                <DIRECTION x="2" y="-1" z="-2" />
            </light>
        </lights>
        
        <materialLibs>
            <materialLib filename="textureModel.mlib" />
        </materialLibs>
        
    </assets>
    
    <pipelines>
    
        <pipeline name="textures" >
        
            <pass class="default" name="texture" >
                <scenes>
                    <scene name="Sphere" />
                </scenes>
                <camera name="MainCamera" />
                <lights>
                    <light name="Sun" />
                </lights>
                <materialMaps>
                    <map fromMaterial="*" toMaterial="textured" toLibrary="Texture" />
                </materialMaps>
                
            </pass>
        </pipeline>
        
    </pipelines>
</project>
```

</details>

<details>
<summary><b>Original:</b> <code>textureModel.mlib</code> (76 linhas)</summary>

```xml
<?xml version="1.0" ?>
<materialLib name="Texture">

    <textures>
        <texture name="earth" filename="otherTextures/earthDay.jpg" mipmap="true" />
        <texture name="spec" filename="otherTextures/earthSpec.jpg" mipmap="true" />
        <texture name="eNight" filename="otherTextures/earthLights.jpg" mipmap="true" />
        <texture name="eClouds" filename="otherTextures/earthClouds.jpg" mipmap="true" />
    </textures>

    <shaders>

        <shader name="textured">
            <vs>
                <file name = "shaders/texture.vert" />
            </vs>
            <ps>
                <file name = "shaders/texture.frag" />
                <file name = "shaders/noise.frag" />
            </ps>
        </shader>
    </shaders>                  
    
    <materials>
        <material name="textured">
            <shader name="textured">
                <values>
                    <valueof uniform="m_pvm" 
                             type="RENDERER" 
                             component="PROJECTION_VIEW_MODEL" />
                     
                    <valueof uniform="m_viewModel" 
                             type="RENDERER"  
                             component="VIEW_MODEL" />
                     
                    <valueof uniform="m_view" 
                             type="RENDERER" 
                             component="VIEW" />
                     
                    <valueof uniform="m_normal" 
                             type="RENDERER" 
                             component="NORMAL" />
                     
                    <valueof uniform="l_dir"
                             type="LIGHT" 
                             component="DIRECTION" id="0"/>
                    <valueof uniform="time"
                             type="RENDERER" 
                             component="TIMER" />

                    <valueof uniform="texEarth"
                             type="TEXTURE_BINDING" context="CURRENT"
                             component="UNIT" id="0" />
                    <valueof uniform="textSpec"
                             type="TEXTURE_BINDING" context="CURRENT"
                             component="UNIT" id="1" />
                    <valueof uniform="texNight"
                             type="TEXTURE_BINDING" context="CURRENT"
                             component="UNIT" id="2" />
                    <valueof uniform="texClouds"
                             type="TEXTURE_BINDING" context="CURRENT"
                             component="UNIT" id="3" />
                    

                </values>
            </shader>
            <textures>
                <texture name="earth" UNIT="0" />
                <texture name="spec" UNIT="1" />
                <texture name="eNight" UNIT="2" />
                <texture name="eClouds" UNIT="3" />
            </textures>
        </material>
    </materials>
    
</materialLib>
```

</details>

<details>
<summary><b>Original:</b> <code>shaders/texture.vert</code> (28 linhas)</summary>

```glsl
#version 330

uniform mat4 m_pvm;
uniform mat4 m_viewModel;
uniform mat4 m_view;
uniform mat3 m_normal;

uniform vec4 l_dir;    // global space

in vec4 position;   // local space
in vec3 normal;     // local space
in vec2 texCoord0;

// the data to be sent to the fragment shader

out vec4 eye;
out vec3 n;
out vec3 ld;
out vec2 textCoord;

void main () {
    textCoord = texCoord0;
    n = normalize(m_normal * normal);
    eye = -(m_viewModel * position);
    ld = normalize(vec3(m_view * -l_dir));

    gl_Position = m_pvm * position; 
}
```

</details>

<details>
<summary><b>Original:</b> <code>shaders/texture.frag</code> (70 linhas)</summary>

```glsl
#version 330

uniform float shininess = 128;
uniform sampler2D texEarth;
uniform sampler2D textSpec;
uniform sampler2D texNight;
uniform sampler2D texClouds;
uniform float time;

in  vec4 eye;
in  vec3 n;
in  vec3 ld;
in vec2 textCoord;
out vec4 colorOut;

float snoise(vec4);

float perlinNoise(vec4 p){
    float c = 0.0, amp = 1.0;
    float freq = 1.0;

    for(int i = 0; i < 5; ++i){
        c = c + snoise(p * freq) * amp;
        amp  *= 0.5;
        freq *= 2.0;
    }
    return c * 0.5 + 0.5;
}

const float PI = 3.14159;

void main() {

    vec4 eColor = texture(texEarth, textCoord);
    vec4 eSpec = texture(textSpec, textCoord);
    vec4 eNight = texture(texNight, textCoord);
    //vec4 eclouds = texture(texClouds, textCoord);
    // set the specular term to black
    vec2 t = vec2(2 * PI * textCoord.s + time * 0.00002, PI * (textCoord - 0.5) );
    vec3 sc = vec3(sin(t.s) * cos(t.t) , sin(t.t), cos(t.s)*cos(t.t));

    float eclouds = perlinNoise(vec4(textCoord * 32.0, time * 0.0001, 0.0));
    eclouds = smoothstep(0.4,0.9,eclouds);
    vec4 spec = vec4(0.0);

    // normalize both input vectors
    vec3 nd = normalize(n);
    vec3 e = normalize(vec3(eye));

    float intensity = max(dot(nd,ld), 0.0);

    // if the vertex is lit compute the specular color
    if (intensity > 0.0) {
        // compute the half vector
        vec3 h = normalize(ld + e); 
        // compute the specular intensity
        float intSpec = max(dot(h,nd), 0.0);
        // compute the specular term into spec
        // the spec map is white on the oceans and black on land
        // earthSpec.jpg has a single channel, so only .r holds the value
        spec = vec4(eSpec.r) * pow(intSpec,shininess);
    }
    vec4 colorDay = max(intensity * eColor + spec, eColor * 0.25) * (1.0 - eclouds) + eclouds;
    vec4 colorNight = eNight * (1.0 - eclouds);

    float f = smoothstep(0.0, 0.1, intensity);
    colorOut = mix(colorNight, colorDay,f);


}
```

</details>

<details>
<summary><b>Original:</b> <code>shaders/noise.frag</code> (132 linhas)</summary>

Biblioteca externa de ruído (Ashima Arts, licença MIT), usada pelo `texture.frag`. Não é escrita pelos alunos: o compilador NSL copia-a sem alterações quando um material usa `layer clouds noise`.

```glsl
//
// Description : Array and textureless GLSL 2D/3D/4D simplex 
//               noise functions.
//      Author : Ian McEwan, Ashima Arts.
//  Maintainer : stegu
//     Lastmod : 20110822 (ijm)
//     License : Copyright (C) 2011 Ashima Arts. All rights reserved.
//               Distributed under the MIT License. See LICENSE file.
//               https://github.com/ashima/webgl-noise
//               https://github.com/stegu/webgl-noise
// 

vec4 mod289(vec4 x) {
  return x - floor(x * (1.0 / 289.0)) * 289.0; }

float mod289(float x) {
  return x - floor(x * (1.0 / 289.0)) * 289.0; }

vec4 permute(vec4 x) {
     return mod289(((x*34.0)+10.0)*x);
}

float permute(float x) {
     return mod289(((x*34.0)+10.0)*x);
}

vec4 taylorInvSqrt(vec4 r)
{
  return 1.79284291400159 - 0.85373472095314 * r;
}

float taylorInvSqrt(float r)
{
  return 1.79284291400159 - 0.85373472095314 * r;
}

vec4 grad4(float j, vec4 ip)
  {
  const vec4 ones = vec4(1.0, 1.0, 1.0, -1.0);
  vec4 p,s;

  p.xyz = floor( fract (vec3(j) * ip.xyz) * 7.0) * ip.z - 1.0;
  p.w = 1.5 - dot(abs(p.xyz), ones.xyz);
  s = vec4(lessThan(p, vec4(0.0)));
  p.xyz = p.xyz + (s.xyz*2.0 - 1.0) * s.www; 

  return p;
  }
                        
// (sqrt(5) - 1)/4 = F4, used once below
#define F4 0.309016994374947451

float snoise(vec4 v)
  {
  const vec4  C = vec4( 0.138196601125011,  // (5 - sqrt(5))/20  G4
                        0.276393202250021,  // 2 * G4
                        0.414589803375032,  // 3 * G4
                       -0.447213595499958); // -1 + 4 * G4

// First corner
  vec4 i  = floor(v + dot(v, vec4(F4)) );
  vec4 x0 = v -   i + dot(i, C.xxxx);

// Other corners

// Rank sorting originally contributed by Bill Licea-Kane, AMD (formerly ATI)
  vec4 i0;
  vec3 isX = step( x0.yzw, x0.xxx );
  vec3 isYZ = step( x0.zww, x0.yyz );
//  i0.x = dot( isX, vec3( 1.0 ) );
  i0.x = isX.x + isX.y + isX.z;
  i0.yzw = 1.0 - isX;
//  i0.y += dot( isYZ.xy, vec2( 1.0 ) );
  i0.y += isYZ.x + isYZ.y;
  i0.zw += 1.0 - isYZ.xy;
  i0.z += isYZ.z;
  i0.w += 1.0 - isYZ.z;

  // i0 now contains the unique values 0,1,2,3 in each channel
  vec4 i3 = clamp( i0, 0.0, 1.0 );
  vec4 i2 = clamp( i0-1.0, 0.0, 1.0 );
  vec4 i1 = clamp( i0-2.0, 0.0, 1.0 );

  //  x0 = x0 - 0.0 + 0.0 * C.xxxx
  //  x1 = x0 - i1  + 1.0 * C.xxxx
  //  x2 = x0 - i2  + 2.0 * C.xxxx
  //  x3 = x0 - i3  + 3.0 * C.xxxx
  //  x4 = x0 - 1.0 + 4.0 * C.xxxx
  vec4 x1 = x0 - i1 + C.xxxx;
  vec4 x2 = x0 - i2 + C.yyyy;
  vec4 x3 = x0 - i3 + C.zzzz;
  vec4 x4 = x0 + C.wwww;

// Permutations
  i = mod289(i); 
  float j0 = permute( permute( permute( permute(i.w) + i.z) + i.y) + i.x);
  vec4 j1 = permute( permute( permute( permute (
             i.w + vec4(i1.w, i2.w, i3.w, 1.0 ))
           + i.z + vec4(i1.z, i2.z, i3.z, 1.0 ))
           + i.y + vec4(i1.y, i2.y, i3.y, 1.0 ))
           + i.x + vec4(i1.x, i2.x, i3.x, 1.0 ));

// Gradients: 7x7x6 points over a cube, mapped onto a 4-cross polytope
// 7*7*6 = 294, which is close to the ring size 17*17 = 289.
  vec4 ip = vec4(1.0/294.0, 1.0/49.0, 1.0/7.0, 0.0) ;

  vec4 p0 = grad4(j0,   ip);
  vec4 p1 = grad4(j1.x, ip);
  vec4 p2 = grad4(j1.y, ip);
  vec4 p3 = grad4(j1.z, ip);
  vec4 p4 = grad4(j1.w, ip);

// Normalise gradients
  vec4 norm = taylorInvSqrt(vec4(dot(p0,p0), dot(p1,p1), dot(p2, p2), dot(p3,p3)));
  p0 *= norm.x;
  p1 *= norm.y;
  p2 *= norm.z;
  p3 *= norm.w;
  p4 *= taylorInvSqrt(dot(p4,p4));

// Mix contributions from the five corners
  vec3 m0 = max(0.6 - vec3(dot(x0,x0), dot(x1,x1), dot(x2,x2)), 0.0);
  vec2 m1 = max(0.6 - vec2(dot(x3,x3), dot(x4,x4)            ), 0.0);
  m0 = m0 * m0;
  m1 = m1 * m1;
  return 49.0 * ( dot(m0*m0, vec3( dot( p0, x0 ), dot( p1, x1 ), dot( p2, x2 )))
               + dot(m1*m1, vec2( dot( p3, x3 ), dot( p4, x4 ) ) ) ) ;

  }

  
//-------------------------------------------------------------------------------
```

</details>

A construção `layer` tem sempre a mesma forma, `layer <papel> <origem>`, em que a origem é uma textura declarada ou, no caso das nuvens, `noise` (ruído gerado no shader). No passe, `lighting X for all` (exemplos 1 e 2) mantém as cores do modelo e troca apenas a forma de iluminar, enquanto `material X for all` substitui o material inteiro.

### 1.7 Como o compilador gera os shaders

O utilizador não escreve GLSL, mas os shaders continuam a existir: é o compilador que os escreve. Para cada material, o compilador monta um par `.vert` / `.frag` a partir de **blocos de código pré-definidos**, escolhidos pelo modo de iluminação e pelas camadas. Os blocos são os próprios shaders das aulas 2 e 3, partidos em pedaços reutilizáveis.

**Correspondência entre a NSL e o GLSL gerado:**

| Na NSL | O que o compilador põe nos shaders | Origem nas aulas |
|---|---|---|
| (sempre) | vertex shader base: transforma a posição (`m_pvm * position`) e passa a normal e a direção da luz para o espaço da câmara | todos |
| `lighting none` | fragment: a cor final é a cor base, sem luz | — |
| `lighting per_vertex` | vertex: calcula `intensity = max(0, dot(n, l))` e a cor `intensity * diffuse`; fragment: só copia a cor recebida | `simple.vert` / `simple.frag` (aula 2) |
| `lighting phong { shininess s; }` | vertex: passa `n`, `l` e o vetor para o olho `e`; fragment: calcula a intensidade em cada píxel e soma o brilho especular `pow(max(0, dot(h, n)), s)`, com `h` o vetor a meio entre `l` e `e` | `v4.vert` / `v4.frag` (aula 2) |
| `lighting toon { levels k; }` | como `per_vertex`, mas arredonda `intensity` para `k` patamares | variante do `v4.vert` com 4 patamares (aula 2) |
| `layer day t` | vertex: passa as coordenadas de textura; fragment: a cor base passa a ser `texture(t, uv)` em vez de `diffuse` | `texture.vert` / `texture.frag` (aula 3) |
| `layer specular t` | fragment: o brilho especular é multiplicado por `texture(t, uv).r` (só brilha onde a imagem é clara) | `texture.frag` (aula 3) |
| `layer night t` | fragment: `mix(cor_noite, cor_dia, smoothstep(0.0, 0.1, intensity))`, para as luzes da cidade aparecerem só no lado escuro | `texture.frag` (aula 3) |
| `layer clouds noise speed v` | junta o ficheiro de ruído ao fragment shader; fragment: calcula nuvens com `perlinNoise(..., time * v)` e mistura-as com branco | `noise.frag` + `texture.frag` (aula 3) |

**Um shader por combinação, não por material.** O shader só depende da *forma* do material: o modo de iluminação e as camadas que tem. Não depende dos *valores*: a cor, a `shininess`, a direção da luz ou a posição da câmara não ficam escritas no GLSL. Ficam no `.mlib` e o Nau entrega-as ao shader através de uniforms, tal como já acontece nas aulas. Por isso, um material amarelo e um azul, ambos `phong`, usam o **mesmo** `phong.frag`, e mudar a luz de sítio não gera shader nenhum. O número de shaders diferentes é pequeno e limitado: no máximo um por cada combinação de modo e camadas que apareça no programa.

**O que é gerado.** Para cada combinação usada, o compilador escreve um par de shaders com um nome derivado dela (`phong.vert`/`phong.frag`, `phong_day_specular_night_clouds.frag`, ...). Os blocos são concatenados pela ordem da tabela, e cada bloco declara os uniforms de que precisa. O compilador junta essas declarações, usando sempre os mesmos nomes (`m_pvm`, `m_view`, `m_normal`, `l_dir`, `diffuse`, `shininess`, `time`, `tex0`, `tex1`, ...). Para cada uniform escreve no `.mlib` a linha `<valueof>` correspondente, e para cada camada uma unidade de textura (0, 1, 2, ...). Como os dois lados (o shader e o `.mlib`) saem do mesmo sítio, os nomes coincidem sempre.

Por exemplo, para o material

```
material amarelo {
    diffuse rgb(1.0, 0.87, 0.0);
    lighting phong { shininess 100; }
}
```

o compilador vê a combinação "`phong`, sem camadas" e gera (uma única vez) um `phong.frag` equivalente ao `v4.frag` da aula 2:

```glsl
#version 460
in vec3 n, l, e;
uniform vec4  diffuse;
uniform float shininess;
out vec4 color;

void main() {
    vec3 nn = normalize(n);
    float intensity = max(0.0, dot(nn, normalize(l)));
    vec4 specular = vec4(0.0);
    if (intensity > 0.0) {
        vec3 h = normalize(normalize(e) + normalize(l));
        specular = vec4(1.0) * pow(max(0.0, dot(h, nn)), shininess);
    }
    color = max(0.25, intensity) * diffuse + specular;
}
```

A cor e o brilho do `amarelo` vão para o `.mlib`, que diz ao Nau para os pôr nos uniforms `diffuse` e `shininess`:

```xml
<material name="amarelo">
    <color>
        <DIFFUSE r="1.0" g="0.87" b="0.0" a="1.0" />
        <SHININESS value="100" />
    </color>
    <shader name="phong">
        <values>
            <valueof uniform="m_pvm"     type="RENDERER" component="PROJECTION_VIEW_MODEL" />
            <valueof uniform="m_view"    type="RENDERER" component="VIEW" />
            <valueof uniform="m_normal"  type="RENDERER" component="NORMAL" />
            <valueof uniform="l_dir"     type="LIGHT"    component="DIRECTION" id="0" />
            <valueof uniform="diffuse"   type="COLOR"    component="DIFFUSE" />
            <valueof uniform="shininess" type="COLOR"    component="SHININESS" />
        </values>
    </shader>
</material>
```

Um segundo material `azul` com `lighting phong` gera outro bloco `<material>` igual a este, só com outros números em `<DIFFUSE>`, e aponta para o mesmo `<shader name="phong">`. Este é o mesmo mecanismo que o `simple.mlib` da aula 2 já usa. A direção da luz também nunca entra no shader: vem da declaração `light` e é entregue em `l_dir`.

**Ligação com a gramática de atributos.** O conjunto de blocos de cada material, e por isso o conjunto de uniforms e de unidades de textura, pode ser descrito como um **atributo sintetizado**. Sobe das declarações `lighting` e `layer` até ao nó `material`, e é a partir dele que se gera o `.mlib`. As regras de coerência da secção 1.5 (por exemplo, `layer specular` só com `phong`) garantem que os blocos escolhidos encaixam uns nos outros.